# Music Flow Matching — Mixed Dataset Training (Colab GPU)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your local `data/nsynth-valid/` folder to Google Drive at:
   `MyDrive/music-flow-matching/data/nsynth-valid/`
   (contains `audio/` and `examples.json` — ~3.4 GB, one-time upload)

VocalSet is downloaded automatically inside Colab — no manual upload needed.

**What this notebook does:**
1. Fixes numpy (force-reinstall) → **requires a runtime restart after Cell 1**
2. Clones repo from GitHub + installs dependencies
3. Mounts Drive, links NSynth data
4. Downloads VocalSet (2.1 GB) directly into Colab
5. Recomputes MEL_MEAN/MEL_STD on the mixed dataset
6. Retrains the VAE (~10 min on T4)
7. Retrains the flow model (~15 min on T4)
8. Copies checkpoints back to Drive for download

In [ ]:
# ── Cell 1: Fix numpy FIRST — must restart runtime after this cell ────────────
# Colab's default numpy gets corrupted by partial pip reinstalls.
# Force a clean reinstall here, then restart so Python loads it fresh.
import subprocess
subprocess.run(['pip', 'install', '-q', '--force-reinstall', 'numpy>=2.0'], check=True)
print()
print('=' * 60)
print('numpy reinstalled. NOW GO TO: Runtime → Restart session')
print('After restart, run from Cell 2 onwards — do NOT re-run Cell 1.')
print('=' * 60)

In [ ]:
# ── Cell 2: Check GPU ─────────────────────────────────────────────────────────
import torch
import numpy as np
print('numpy version:', np.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# ── Cell 3: Clone repo + install dependencies ─────────────────────────────────
import os

REPO_URL = 'https://github.com/NeelN86/music-flow-matching'
REPO_DIR = '/content/music-flow-matching'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working dir:', os.getcwd())

# numpy is already fixed — pip will see it satisfies numpy>=2.0 and skip it
!pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Mount Google Drive + link NSynth ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE        = '/content/drive/MyDrive/music-flow-matching'
NSYNTH_DIR        = os.path.join(DRIVE_BASE, 'data/nsynth-valid')
CHECKPOINTS_DRIVE = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(CHECKPOINTS_DRIVE, exist_ok=True)

if not os.path.isdir(NSYNTH_DIR):
    raise RuntimeError(
        f'NSynth not found at {NSYNTH_DIR}\n'
        'Upload data/nsynth-valid/ to Google Drive first.'
    )

os.makedirs('data', exist_ok=True)
if not os.path.lexists('data/nsynth-valid'):
    os.symlink(NSYNTH_DIR, 'data/nsynth-valid')
    print(f'Linked NSynth: {NSYNTH_DIR} → data/nsynth-valid')
else:
    print('data/nsynth-valid already linked')

import glob
n = len(glob.glob('data/nsynth-valid/audio/*.wav'))
print(f'NSynth WAVs found: {n}')

In [ ]:
# ── Cell 5: Download VocalSet directly into Colab ─────────────────────────────
VOCALSET_ZIP = '/content/VocalSet.zip'
VOCALSET_DIR = '/content/music-flow-matching/data/vocalset'

real_wavs = [
    p for p in glob.glob(f'{VOCALSET_DIR}/**/*.wav', recursive=True)
    if not os.path.basename(p).startswith('._') and '__MACOSX' not in p
]

if not real_wavs:
    if not os.path.exists(VOCALSET_ZIP):
        print('Downloading VocalSet (2.1 GB)...')
        !wget -q --show-progress -O {VOCALSET_ZIP} \
            'https://zenodo.org/records/1193957/files/VocalSet.zip?download=1'
        print('Download complete.')
    print('Unzipping...')
    os.makedirs(VOCALSET_DIR, exist_ok=True)
    !unzip -q {VOCALSET_ZIP} -d {VOCALSET_DIR}
    !rm -rf {VOCALSET_DIR}/__MACOSX {VOCALSET_ZIP}
    print('Unzip complete.')
else:
    print('VocalSet already extracted.')

real_wavs = [
    p for p in glob.glob(f'{VOCALSET_DIR}/**/*.wav', recursive=True)
    if not os.path.basename(p).startswith('._') and '__MACOSX' not in p
]
print(f'VocalSet WAVs found: {len(real_wavs)}')

In [ ]:
# ── Cell 6: Recompute mel stats on mixed dataset ──────────────────────────────
import sys, re
sys.path.insert(0, '.')
from src.data import compute_mel_stats_mixed

print('Computing mel stats (samples from NSynth + VocalSet)...')
mean, std = compute_mel_stats_mixed(
    audio_dirs=['data/nsynth-valid/audio', VOCALSET_DIR],
    max_samples_per_dir=2000,
)
print(f'MEL_MEAN = {mean:.4f}')
print(f'MEL_STD  = {std:.4f}')

config_path = 'src/config.py'
with open(config_path) as f:
    cfg = f.read()
cfg = re.sub(r'MEL_MEAN: float = [\d.+-]+', f'MEL_MEAN: float = {mean:.4f}', cfg)
cfg = re.sub(r'MEL_STD: float = [\d.+-]+',  f'MEL_STD: float = {std:.4f}',  cfg)
with open(config_path, 'w') as f:
    f.write(cfg)

import importlib, src.config
importlib.reload(src.config)
print('config.py updated and reloaded.')

In [ ]:
# ── Cell 7: Train VAE on mixed dataset (~10 min on T4) ────────────────────────
!python scripts/train_vae.py \
    --data_dir data/nsynth-valid \
    --extra_audio_dirs {VOCALSET_DIR} \
    --epochs 30 \
    --batch_size 64 \
    --beta 0.001 \
    --beta_warmup 5 \
    --num_workers 2 \
    --checkpoint checkpoints/audio_vae.pt

In [ ]:
# ── Cell 8: Train flow model on mixed dataset (~15 min on T4) ─────────────────
import os
stale = 'outputs/latents_cache.npz'
if os.path.exists(stale):
    os.remove(stale)
    print('Deleted stale latents_cache.npz')

!python scripts/train_flow.py \
    --data_dir data/nsynth-valid \
    --extra_audio_dirs {VOCALSET_DIR} \
    --steps 30000 \
    --batch_size 256 \
    --num_workers 2 \
    --checkpoint_out checkpoints/flow_model.pt

In [ ]:
# ── Cell 9: Copy checkpoints to Drive ────────────────────────────────────────
import shutil, os

for fname in ['audio_vae.pt', 'flow_model.pt']:
    dst = os.path.join(CHECKPOINTS_DRIVE, fname)
    shutil.copy2(f'checkpoints/{fname}', dst)
    print(f'Saved {fname} → {dst}')

shutil.copy2('src/config.py', os.path.join(CHECKPOINTS_DRIVE, 'config_mixed.py'))
print('Saved config_mixed.py → Drive')
print()
print('All done! See below for next steps.')

## After Colab finishes — local steps

Download these three files from `MyDrive/music-flow-matching/checkpoints/` in Google Drive:

1. `audio_vae.pt` → replace local `checkpoints/audio_vae.pt`
2. `flow_model.pt` → replace local `checkpoints/flow_model.pt`
3. `config_mixed.py` → open it, copy `MEL_MEAN` and `MEL_STD` into local `src/config.py`

Then:
```
del outputs\latents_cache.npz
python app.py
```
Click **Load latent space** to rebuild the PCA cache with the new VAE.